Ноутбук кодирует русский корпус моделью E. На выходе сохраняются эмбеддинги документов, которые потом используются для оценки метрик.


In [ ]:
!pip -q install transformers accelerate sentencepiece protobuf faiss-cpu

import os, json, time, hashlib, math, gc
from collections import defaultdict, Counter

import numpy as np
import torch
import torch.nn.functional as F
from tqdm.auto import tqdm
from transformers import AutoTokenizer, AutoModel

SOURCE_CORPUS_DIR = "/kaggle/input/datasets/sukiss/corpus-embedings"

E5_MODEL_DIR = "/kaggle/working/e5_multilingual_base_ru_retriever"

OUT_DIR = "/kaggle/working/e5_proxy_encoding"
os.makedirs(OUT_DIR, exist_ok=True)

BATCH_DOCS = 64
PASSAGE_MAX_LEN = 192
DTYPE_SAVE = np.float16

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
assert DEVICE == "cuda"

tokenizer = AutoTokenizer.from_pretrained(E5_MODEL_DIR)
model = AutoModel.from_pretrained(E5_MODEL_DIR)
model.to(DEVICE)
model.eval()

print("Loaded E5 model:", E5_MODEL_DIR)

def average_pool(last_hidden_states, attention_mask):
    mask = attention_mask.unsqueeze(-1).expand(last_hidden_states.size()).float()
    masked = last_hidden_states * mask
    return masked.sum(dim=1) / mask.sum(dim=1).clamp(min=1e-9)

@torch.inference_mode()
def encode_texts(texts, max_len, is_query):
    prefix = "query: " if is_query else "passage: "
    texts = [prefix + x for x in texts]

    batch = tokenizer(
        texts,
        padding=True,
        truncation=True,
        max_length=max_len,
        return_tensors="pt",
    )
    batch = {k: v.to(DEVICE) for k, v in batch.items()}

    out = model(**batch)
    emb = average_pool(out.last_hidden_state, batch["attention_mask"])
    emb = F.normalize(emb, p=2, dim=-1)
    return emb.detach().cpu().numpy()

def load_corpus_jsonl(path):
    docids, texts = [], []

    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            obj = json.loads(line)
            docids.append(str(obj["docid"]))
            texts.append(obj["text"])

    return docids, texts

def encode_corpus(name):
    src_path = os.path.join(SOURCE_CORPUS_DIR, f"{name}_corpus.jsonl")
    out_corpus = os.path.join(OUT_DIR, f"{name}_corpus.jsonl")
    out_docids = os.path.join(OUT_DIR, f"{name}_docids.txt")
    out_emb = os.path.join(OUT_DIR, f"{name}_embeddings.npy")

    assert os.path.exists(src_path), src_path

    if os.path.exists(out_emb) and os.path.exists(out_docids) and os.path.exists(out_corpus):
        print(f"[{name}] already encoded, skipping")
        return

    docids, texts = load_corpus_jsonl(src_path)
    print(f"[{name}] docs:", len(docids))

    with open(out_corpus, "w", encoding="utf-8") as f:
        for d, t in zip(docids, texts):
            f.write(json.dumps({"docid": d, "text": t}, ensure_ascii=False) + "\n")

    with open(out_docids, "w", encoding="utf-8") as f:
        for d in docids:
            f.write(d + "\n")

    first = encode_texts(texts[:1], PASSAGE_MAX_LEN, is_query=False).astype(DTYPE_SAVE)
    dim = first.shape[1]

    emb = np.memmap(out_emb, dtype=DTYPE_SAVE, mode="w+", shape=(len(texts), dim))
    emb[0:1] = first

    t0 = time.time()

    for i in tqdm(range(1, len(texts), BATCH_DOCS), desc=f"encoding {name}"):
        j = min(len(texts), i + BATCH_DOCS)
        emb[i:j] = encode_texts(texts[i:j], PASSAGE_MAX_LEN, is_query=False).astype(DTYPE_SAVE)

    emb.flush()

    print(f"[{name}] saved:")
    print(" corpus:", out_corpus)
    print(" docids:", out_docids)
    print(" emb:", out_emb, "dim:", dim, "MB:", round(os.path.getsize(out_emb) / 1024 / 1024, 2))
    print(" time_s:", round(time.time() - t0, 1))

encode_corpus("main")
encode_corpus("test")

print("DONE:", sorted(os.listdir(OUT_DIR)))
